# Düzenlileştirme

**Titanic** yolcularının hayatta kalma şansını etkileyen faktörlere dair anlayışımızı geliştirelim
- Yorumlaması kolay olan lojistik sınıflandırıcıları kullanacağız
- Bunu daha önce "Karar Bilimi - Lojistik Regresyon" dersinde statsmodels ile yapmıştık
- Hangi özelliklerin alakasız olduğunu / genelleştirilemediğini tespit etmek için `p-değerleri` ve istatistiksel varsayımlar kullanıyorduk
- Bu sefer, eksik/aşırı öğrenme kriterlerine dayalı olarak alakalı/alakasız özellikleri tespit etmek için `düzenlileştirme` kullanacağız
- **Amacımız `L1` ve `L2` cezalarını karşılaştırmak**

## 1. Veriyi sizin için yüklüyor ve ön işleme tabi tutuyoruz

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv")

# the dataset is already one-hot-encoded
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [3]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [4]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Düzenlileştirme olmadan Lojistik Regresyon

❓ Basit bir **düzenlileştirilmemiş** Lojistik Regresyon eğittikten sonra özellikleri önem sırasına göre azalan şekilde sıralayın (yani, eğitim sonrası katsayılara bakın)
- Dikkat: `LogisticRegression` varsayılan olarak cezalandırılmıştır
  - cezayı nasıl kaldıracağınızı öğrenmek için [penalty parametresine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) bakın)
- Model yakınsayana kadar `max_iter`'i daha büyük bir sayıya çıkarın
- Çözücünün durma kriterini ayarlamak için `tol=1e-9` kullanın: gradyanın en büyük bileşeni bundan küçük olduğunda çözücü duracak. Daha yüksek değerlere ayarlarsanız, katsayıların `tol` değeriyle birlikte çok dalgalandığını görürsünüz.

<details>
    <summary>İpucu</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

# X ve y oluşturma
y = data["survived"]
X = data.drop(columns=["survived"])

# MinMaxScale Uygulama
scaler = MinMaxScaler().fit(X)
X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns)

# Düzenlileştirme Olmadan (Unregularized) Lojistik Regresyon
# Not: penalty=None cezalandırmayı kaldırır. 
# max_iter ve tol değerleri soruda istendiği gibi ayarlandı.
model_unreg = LogisticRegression(
    penalty=None, 
    max_iter=10000, 
    tol=1e-9
)
model_unreg.fit(X_scaled, y)

# Özellikleri Önem Sırasına Göre Sıralama (Katsayıların mutlak değerine göre)
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_unreg.coef_[0],
    'Absolute_Coefficient': np.abs(model_unreg.coef_[0])
}).sort_values(by='Absolute_Coefficient', ascending=False)

# Sonuçları Yazdır
print("--- Düzenlileştirilmemiş Model Katsayıları ---")
print(importance_df[['Feature', 'Coefficient']])

# Görsel olarak doğrulamak istersen:
# importance_df.set_index('Feature')['Coefficient'].plot(kind='barh')

--- Düzenlileştirilmemiş Model Katsayıları ---
                    Feature  Coefficient
10   embark_town_Queenstown   -21.410380
11  embark_town_Southampton   -21.014975
9     embark_town_Cherbourg   -20.713605
0                    pclass     5.259139
7               class_Third    -3.812765
6               class_First     3.716370
5                sex_female     2.671879
2                     sibsp    -2.476884
1                       age    -2.196126
4                      fare     1.360205
8                 who_child     1.336358
3                     parch    -0.894275


❓`sex_female` katsayısının değerini sade Türkçe ile nasıl yorumlarsınız?

<details>
    <summary>Cevap</summary>

> "Diğer tüm şeyler eşitken (yaş, bilet sınıfı vb...),
kadın olmak hayatta kalma log-oranlarınızı 2.67 artırır (sizin katsayı değeriniz)"
    
> "Bu veri setinde mevcut olan diğer tüm açıklayıcı faktörleri kontrol ederken,
kadın olmak hayatta kalma oranlarınızı exp(2.67) = 14 kat artırır"

</details>

❓ Modelinize göre hayatta kalma şansını en çok etkileyen özellik hangisidir?  
Aşağıdaki `top_1_feature` listesini bu özelliğin adıyla doldurun

In [8]:
top_1_feature = ["embark_town_Queenstown"]

In [9]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/semih/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/semih/code/Semih0799/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. L2 cezalı Lojistik Regresyon

Aşırı öğrenme olmadan **en önemli özellikleri** bulmak için log-kaybı **L2** terimi ile cezalandırılmış bir **Lojistik model** kullanalım.  
Bu, "Ridge" regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın (katsayılara bakın)
- "Güçlü düzenlileştirilmiş" ile "Sklearn'in varsayılan düzenlileştirme faktöründen daha fazla" demek istiyoruz. 
- Sklearn'in varsayılan değerleri "ölçeklenmiş özellikler" için akılda tutulması gereken çok yararlı büyüklük mertebeleridir

In [10]:
from sklearn.linear_model import LogisticRegression

# 1. Güçlü düzenlileştirilmiş (L2) Lojistik Regresyon
# C=0.01 kullanarak varsayılan (1.0) değerden çok daha güçlü bir ceza uyguluyoruz
model_l2 = LogisticRegression(
    penalty='l2', 
    C=0.01, 
    max_iter=1000, 
    solver='lbfgs'
)
model_l2.fit(X_scaled, y)

# 2. Katsayıları önem sırasına göre sıralama
importance_l2_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_l2.coef_[0],
    'Abs_Coefficient': np.abs(model_l2.coef_[0])
}).sort_values(by='Abs_Coefficient', ascending=False)

print("--- L2 Cezalı (Güçlü Düzenlileştirilmiş) Model Katsayıları ---")
print(importance_l2_df[['Feature', 'Coefficient']])

--- L2 Cezalı (Güçlü Düzenlileştirilmiş) Model Katsayıları ---
                    Feature  Coefficient
5                sex_female     0.614180
7               class_Third    -0.295907
0                    pclass    -0.257020
6               class_First     0.218152
8                 who_child     0.136174
9     embark_town_Cherbourg     0.126071
11  embark_town_Southampton    -0.109161
1                       age    -0.067331
4                      fare     0.051316
3                     parch     0.026725
10   embark_town_Queenstown    -0.024481
2                     sibsp    -0.019400


❓ Modelinize göre hayatta kalma şansını etkileyen ilk 2 özellik hangileridir?  
Aşağıdaki `top_2_features` listesini bu özelliklerin adlarıyla doldurun

In [11]:
top_2_features = ["sex_female", "class_Third"]

#### 🧪 Kodunuzu aşağıda test edin

In [12]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/semih/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/semih/code/Semih0799/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. L1 cezalı Lojistik Regresyon

Bu sefer, **daha az önemli özellikleri filtrelemek** için log-kaybı **L1** terimi ile cezalandırılmış bir lojistik model kullanacağız.  
Bu, **Lasso** regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın

In [14]:
from sklearn.linear_model import LogisticRegression

# C değerini 0.1 yaparak cezayı biraz hafifletiyoruz
model_l1_v2 = LogisticRegression(
    penalty='l1', 
    C=0.1, 
    solver='liblinear', 
    max_iter=1000
)
model_l1_v2.fit(X_scaled, y)

# Katsayıları tekrar inceleyelim
importance_l1_v2 = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_l1_v2.coef_[0]
})

# Katsayısı tam olarak 0 olanları filtrele
zero_impact_list = importance_l1_v2[importance_l1_v2['Coefficient'] == 0]['Feature'].tolist()
print("Sıfıra inen özellikler:", zero_impact_list)

Sıfıra inen özellikler: ['sibsp', 'parch', 'fare', 'class_First', 'embark_town_Cherbourg', 'embark_town_Queenstown']


❓ L1 modelinize göre hayatta kalma şansı üzerinde kesinlikle hiçbir etkisi olmayan özellikler hangileridir?  
Aşağıdaki `zero_impact_features` listesini bu özelliklerin adlarıyla doldurun; listeye eleman eklemeniz gerekebilir.

- Bunlardan bazılarının düzenlileştirilmemiş modele göre "çok önemli" olduğunu fark ettiniz mi? 
- Bundan sonra doğrusal modellerimizi her zaman düzenlileştireceğiz!

In [15]:
zero_impact_features = ["parch", "sibsp", "embark_town_Queenstown", "embark_town_Southampton"]

#### 🧪 Kodunuzu aşağıda test edin

In [16]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/semih/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/semih/code/Semih0799/S16D5-S-regularization/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_lasso.py::TestLasso::test_zero_impact PASSED                        [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/lasso.pickle

git commit -m 'Completed lasso step'

git push origin master



# 5. Bir adım geri çekilmek

🤯 **Bu katsayılardan bazıları neden başlangıçta bu kadar yüksekti?**

Düzenlileştirme ile kaldırılan üç özelliği düşünelim:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

Üç biniş şehri tabii ki ilişkilidir: ikisinden binmediyseniz, üçüncüsünden binmiş olmalısınız. Yani biliyoruz ki: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

Bu üç özellik **mükemmel çoklu doğrusal bağıntılıdır**!

**Düzenlileştirilmemiş modeller kullanılırken, bu genellikle sayısal kararsızlığa yol açar**, ki burada gördüğümüz tam olarak buydu. Ayrıca böyle bir durumda elde ettiğimiz **katsayılara gerçekten güvenemeyeceğimiz** anlamına gelir.

❗️ Bu üç çoklu doğrusal bağıntılı özellik, `embark_town` kategorik özelliğinin one hot encoding'inden gelir.

Düzenlileştirme sayesinde bu sorunu aştık: üç şehir için katsayıların çok büyük olmasını engelledi. **İşte bu yüzden neredeyse her zaman düzenlileştirme kullanacağız.**

🔍 **Başlangıçta ayarladığımız `tol` parametresini hatırlıyor musunuz?**

Düzenlileştirmenin ekstra bir bonusu da `tol` ayarlamanın daha az önemli hale gelmesi: `1e-2` ve `1e-9` arasında herhangi bir değere ayarlayabilirsiniz ve katsayılar neredeyse hiç değişmez! 💪

**🏁 Tebrikler! Not defterinizi commit etmeyi ve push etmeyi unutmayın**